# Week 4 Assignment · The Compression Gauntlet: half → 8-bit → 4-bit
### Build Custom AI — SarasAI · *ungraded practice*

Compression claims are cheap — **measurements are the assignment**. You take an Analyst model,
produce **three precision variants** (half, 8-bit, 4-bit), and run every one through the same
gauntlet: memory, latency, quality on the frozen set — ending in a gate verdict you committed
to *before* seeing any number.

| Part | Tasks | What you build |
|---|---|---|
| 1 · One standalone model | 1–3 | frozen-set contract check · adapter fold-in · memory accounting |
| 2 · Three precision arms | 4–6 | 8-bit + 4-bit reloads · the memory comparison |
| 3 · Speed, honestly | 7–8 | `bench()` with true token counts · all three arms benchmarked |
| 4 · The gate | 9–11 | format re-score · embedding **content score** · verdict + deliverable table |
| 5 · Ship it | 12 | GGUF export plan + mergekit config (on paper, precisely) |

⏱ ~60–75 min on a T4. Fully standalone: Week-3 artifacts are picked up if present,
with clearly-labeled fallbacks if not.

> 📘 **This is the SOLUTION notebook.** One reference implementation — yours may differ and
> be equally correct.

---
## Setup (given)

In [ ]:
!pip install -q "transformers>=4.50" "peft>=0.14" bitsandbytes accelerate sentence-transformers

In [ ]:
import torch, time, json, os, re
import numpy as np

assert torch.cuda.is_available(), "This assignment needs a GPU (T4 is enough)."
BF16_OK = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("dtype:", DTYPE)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE_ID)

def chat(model, user_text, max_new_tokens=300):
    inputs = tok.apply_chat_template([{"role": "user", "content": user_text}],
        add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

---
## Part 1 · One standalone model

### ✏️ Task 1 — load the frozen set — and verify its contract

> 💡 **Hint:** os.path.exists → json.load, else FALLBACK. The assert is the actual lesson: validate inputs to your eval.

In [ ]:
# ─── Task 1 · SOLUTION ───
FALLBACK = [{"input": ("Defect note from visual inspection:\n"
                       f"{d.capitalize()} observed on 3 units of {p}, batch F-{4000+i}.\n\n"
                       "Retrieved evidence:\n- review: customer reports the same issue\n"
                       "- spec: component rated for standard household use\n\n"
                       "Write a root cause analysis report."),
             "output": "## ROOT CAUSE ANALYSIS\n**Product:** " + p +
                       "\n**Defect:** " + d + "\n**Severity:** MEDIUM\n**Evidence:** review.\n"
                       "**Probable cause:** Component variation.\n**Recommended action:** Audit batch."}
            for i, (p, d) in enumerate([("KT-2000 kettle", "cracked lid hinge"),
                                        ("TS-400 toaster", "lever sticking"),
                                        ("BL-900 blender", "jar seal leak"),
                                        ("MW-70 microwave", "door latch failure"),
                                        ("KT-2000 kettle", "seal degradation"),
                                        ("BL-900 blender", "blade wobble")])]

if os.path.exists("week3_test_rows.json"):
    test_rows = json.load(open("week3_test_rows.json"))
    print("using YOUR Week-3 frozen set")
else:
    test_rows = FALLBACK
    print("using the built-in fallback set")

assert all(isinstance(r.get("input"), str) and r["input"]
           and isinstance(r.get("output"), str) and r["output"] for r in test_rows), \
    "broken contract: every row needs non-empty input AND output"
test_inputs = [r["input"] for r in test_rows]
print(len(test_rows), "frozen rows 🔒")

### ✏️ Task 2 — fold the adapter into the base

> 💡 **Hint:** PeftModel.from_pretrained(base_model, adapter_dir).merge_and_unload() — one line, two calls.

In [ ]:
# ─── Task 2 · SOLUTION ───
from peft import PeftModel

analyst = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE, device_map="auto")

if os.path.exists("analyst-lora-adapter"):
    analyst = PeftModel.from_pretrained(analyst, "analyst-lora-adapter").merge_and_unload()
    print("adapter folded in — this is YOUR Analyst")
else:
    print("no adapter found — plain base (fallback); every step below is identical")

analyst.save_pretrained("analyst-standalone")
tok.save_pretrained("analyst-standalone")

### ✏️ Task 3 — memory accounting — from parameter counts, then verified

> 💡 **Hint:** fp16/bf16 = 2 bytes per parameter. The measurement should land within ~10% of your prediction.

In [ ]:
# ─── Task 3 · SOLUTION ───
predicted_gb = analyst.num_parameters() * 2 / 1e9

measured_gb = analyst.get_memory_footprint() / 1e9

print(f"predicted: {predicted_gb:.2f} GB   measured: {measured_gb:.2f} GB")

In [ ]:
# ── self-check ──
assert abs(predicted_gb - measured_gb) / measured_gb < 0.15, \
    "prediction and measurement disagree by >15% — check your arithmetic"
print("✅ you can now size ANY model from its parameter count — before downloading it")

---
## Part 2 · Three precision arms

The precision ladder from the deck, made concrete: the same model at 16, 8, and 4 bits.

### ✏️ Task 4 — the 8-bit arm (LLM.int8)

> 💡 **Hint:** one flag: load_in_8bit=True. (This is Dettmers' LLM.int8 — outlier channels stay fp16.)

In [ ]:
# ─── Task 4 · SOLUTION ───
bnb8 = BitsAndBytesConfig(load_in_8bit=True)
analyst_8bit = AutoModelForCausalLM.from_pretrained("analyst-standalone",
                                                    quantization_config=bnb8, device_map="auto")
print(f"8-bit: {analyst_8bit.get_memory_footprint()/1e9:.2f} GB")

### ✏️ Task 5 — the 4-bit arm (NF4)

> 💡 **Hint:** four kwargs — you've typed them twice this course already.

In [ ]:
# ─── Task 5 · SOLUTION ───
bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
analyst_4bit = AutoModelForCausalLM.from_pretrained("analyst-standalone",
                                                    quantization_config=bnb4, device_map="auto")
print(f"4-bit: {analyst_4bit.get_memory_footprint()/1e9:.2f} GB")

### ✏️ Task 6 — the memory ladder, in one printout

> 💡 **Hint:** ratio = half's footprint / this arm's footprint.

In [ ]:
# ─── Task 6 · SOLUTION ───
ARMS = [("half", analyst), ("8-bit", analyst_8bit), ("4-bit", analyst_4bit)]

half_gb = analyst.get_memory_footprint() / 1e9
for name, m in ARMS:
    g = m.get_memory_footprint() / 1e9
    print(f"{name:6} {g:5.2f} GB   {half_gb/g:4.1f}x")

In [ ]:
# ── self-check: the ladder must be monotonic ──
gbs = [m.get_memory_footprint() for _, m in ARMS]
assert gbs[0] > gbs[1] > gbs[2], "memory should strictly shrink down the ladder"
print("✅ 16 > 8 > 4 bits — memory savings are the GUARANTEED part of quantization")

---
## Part 3 · Speed, honestly measured

Two classic benchmark bugs to dodge: trusting a single run (variance), and dividing by
`max_new_tokens` when the model stopped early at EOS.

### ✏️ Task 7 — `bench()` — percentiles + true token counts

> 💡 **Hint:** p95 is what users feel — SLOs are written against it, not the average.

In [ ]:
# ─── Task 7 · SOLUTION ───
def bench(model, prompt, n_runs=6, max_new_tokens=120):
    times, tokens = [], []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        out_text = chat(model, prompt, max_new_tokens=max_new_tokens)
        times.append(time.perf_counter() - t0)
        tokens.append(len(tok.encode(out_text)))
    t = np.array(times)
    return {"p50_s": round(float(np.percentile(t, 50)), 2),
            "p95_s": round(float(np.percentile(t, 95)), 2),
            "tok_per_s": round(float(np.mean(tokens) / np.mean(t)), 1)}

print(bench(analyst, test_inputs[0], n_runs=3))

### ✏️ Task 8 — run the gauntlet

> 💡 **Hint:** a dict comprehension over ARMS.

In [ ]:
# ─── Task 8 · SOLUTION ───
speed = {name: bench(m, test_inputs[0]) for name, m in ARMS}

for name, stats in speed.items():
    print(f"{name:6} {stats}")

In [ ]:
# ── read your own numbers (given) ──
print("""On a T4, don't be shocked if 8-bit and 4-bit are NOT faster than half precision —
bitsandbytes dequantizes on the fly, and the session's verified finding applies:
quantization gains are hardware- and engine-dependent. Memory savings are guaranteed;
speed is not. The freed memory buys bigger batches and cheaper GPUs — that's the real win.""")

---
## Part 4 · The gate

**Commit before you look.** Threshold: neither metric may drop more than **5 points** vs. the
half-precision arm. Format checks structure; the **content score** (embedding similarity to
the reference reports) catches what format can't — intact headers around degraded analysis.

### ✏️ Task 9 — format re-score across the ladder

> 💡 **Hint:** a dict comprehension: {name: [chat(m, x, max_new_tokens=200) for x in test_inputs] for name, m in ARMS}. This is the slow cell — 18-30 generations (rows x 3 arms); the 8-bit arm is the slowest.

In [ ]:
# ─── Task 9 · SOLUTION ───
REQUIRED = ["## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
            "**Evidence:**", "**Probable cause:**", "**Recommended action:**"]

def format_ok(t):
    return all(h in t for h in REQUIRED) and bool(
        re.search(r"\*\*Severity:\*\*\s*(LOW|MEDIUM|HIGH)", t))

outputs = {name: [chat(m, x, max_new_tokens=200) for x in test_inputs] for name, m in ARMS}

rate = lambda outs: sum(map(format_ok, outs)) / len(outs)
for name in outputs:
    print(f"{name:6} format adherence: {rate(outputs[name]):.0%}")

### ✏️ Task 10 — the content score

> 💡 **Hint:** (gen * refs).sum(axis=1) = row-wise cosine on normalized vectors; then .mean().

In [ ]:
# ─── Task 10 · SOLUTION ───
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
refs = embedder.encode([r["output"] for r in test_rows], normalize_embeddings=True)

def content_score(outs):
    """Mean cosine similarity between generated reports and the reference reports."""
    gen = embedder.encode(outs, normalize_embeddings=True)
    return float((gen * refs).sum(axis=1).mean())

for name in outputs:
    print(f"{name:6} content score: {content_score(outputs[name]):.3f}")

### ✏️ Task 11 — verdict + the deliverable table

> 💡 **Hint:** drop in points = (base_rate - arm_rate) * 100. The table is the graded increment's core deliverable — practice its shape now.

In [ ]:
# ─── Task 11 · SOLUTION ───
THRESHOLD_PTS = 5

base_fmt, base_con = rate(outputs["half"]), content_score(outputs["half"])
print(f"{'arm':6} {'GB':>6} {'p50':>6} {'p95':>6} {'tok/s':>7} {'format':>8} {'content':>9}  verdict")
for name, m in ARMS:
    g = m.get_memory_footprint() / 1e9
    s = speed[name]
    f, c = rate(outputs[name]), content_score(outputs[name])
    if name == "half":
        verdict = "(baseline)"
    else:
        fmt_drop, con_drop = (base_fmt - f) * 100, (base_con - c) * 100
        ok = fmt_drop <= THRESHOLD_PTS and con_drop <= THRESHOLD_PTS
        verdict = "✅ ship" if ok else "❌ REJECT"
    print(f"{name:6} {g:6.2f} {s['p50_s']:6} {s['p95_s']:6} {s['tok_per_s']:7} "
          f"{f:8.0%} {c:9.3f}  {verdict}")

In [ ]:
# ── self-check ──
assert set(outputs) == {"half", "8-bit", "4-bit"} and all(len(v) == len(test_rows) for v in outputs.values())
print("✅ full gauntlet complete: 3 arms x memory + latency + 2 quality metrics + gate verdict")

---
## Part 5 · Ship it (on paper, precisely)

### ✏️ Task 12 — the GGUF plan + a mergekit config

> 💡 **Hint:** convert produces an f16 .gguf; llama-quantize (which you must BUILD first — that's the cmake step) shrinks it to q4_k_m.

In [ ]:
# ─── Task 12 · SOLUTION ───
GGUF_STEPS = """
git clone --depth 1 https://github.com/ggml-org/llama.cpp
pip install -r llama.cpp/requirements.txt
python llama.cpp/convert_hf_to_gguf.py ./analyst-standalone --outfile analyst-f16.gguf
cmake -B llama.cpp/build llama.cpp
cmake --build llama.cpp/build --target llama-quantize -j
./llama.cpp/build/bin/llama-quantize analyst-f16.gguf analyst-q4_k_m.gguf q4_k_m
"""

MERGEKIT_YAML = """
models:
  - model: ./analyst-standalone
    parameters: {weight: 0.6, density: 0.6}
  - model: org/second-finetune-of-qwen2.5-1.5b
    parameters: {weight: 0.4, density: 0.6}
merge_method: ties
base_model: Qwen/Qwen2.5-1.5B-Instruct
dtype: float16
"""

ACCEPTANCE_TEST = ("The merged model must RE-PASS each parent's eval: our frozen-set format + "
                   "content gate AND the second model's own eval. No pass, no merge.")
print(GGUF_STEPS)

---
## Wrap-up · What you practiced

| Task | Skill | Where it goes next |
|---|---|---|
| 1–3 | eval-input validation, adapter fold-in, sizing models from param counts | every deployment |
| 4–6 | 8-bit & 4-bit arms, the memory ladder | choosing a serving precision |
| 7–8 | honest benchmarking (percentiles, true tokens) | your SLO numbers |
| 9–11 | two-metric quality gate + the deliverable table | graded increment 4's core |
| 12 | GGUF pipeline, mergekit config, acceptance test | the ship-it conversation |

**Reflection (2 min):** if 4-bit passed the gate but was *slower* than half on your T4 — when
would you still ship it? (Hint: what does the freed memory buy? Where else could it run?)

*Ungraded — nothing to submit. Graded increment 4 = this discipline on your real Analyst,
plus $/1k economics and the assembled VLM→RAG→Analyst pipeline.*